# Week 3 Thursday Exercise 2 — EDD Trigger Decision Tree

This notebook solves the decision-tree exercise using a synthetic AML customer risk dataset.


In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [2]:
df = pd.read_csv('week3_thursday_edd_synthetic_data.csv')
df


,customer_id,customer_name,customer_type,geography,crr_score_0_100,adverse_media,pep,sanctions,ownership_complexity,cross_border_activity,cash_intensity,expected_activity_mismatch,transaction_spike,sudden_profile_change,current_review_outcome
0,H001,Atlas Import Group,Business,Cross-border,1,1,0,1,1,1,0,1,0,1,EDD
1,H002,Bright Side Retail,Retail,Domestic,0,0,0,0,0,0,0,0,0,0,Standard
2,H003,Cedar Family Office,Private wealth,Cross-border,1,0,1,0,1,1,0,1,0,0,EDD
3,H004,Delta Community Health,Business,Domestic,0,0,0,0,0,0,0,0,0,0,Standard
4,H005,Evergreen Payments LLC,Business,Cross-border,1,1,1,1,1,1,1,1,0,1,EDD
5,H006,Frontier Logistics,Business,Domestic,0,1,0,0,1,0,0,1,0,0,EDD


## Define the target
EDD is triggered when the CRR score is high or when at least one trigger flag is present.


In [3]:
trigger_cols = ['adverse_media','pep','sanctions','ownership_complexity','cross_border_activity','cash_intensity','expected_activity_mismatch','transaction_spike','sudden_profile_change']
df['edd_target'] = ((df['crr_score_0_100'] >= 75) | (df[trigger_cols].sum(axis=1) >= 1)).astype(int)
df[['customer_id','customer_name','crr_score_0_100','edd_target']]


,customer_id,customer_name,crr_score_0_100,edd_target
0,H001,Atlas Import Group,1,1
1,H002,Bright Side Retail,0,0
2,H003,Cedar Family Office,1,1
3,H004,Delta Community Health,0,0
4,H005,Evergreen Payments LLC,1,1
5,H006,Frontier Logistics,0,1


## Train the model
Use the score plus trigger flags as features.


In [4]:
features = ['crr_score_0_100'] + trigger_cols
X = df[features]
y = df['edd_target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
accuracy_score(y_test, pred)


0.5

## Evaluate the tree


In [5]:
print(classification_report(y_test, pred, zero_division=0))
print(confusion_matrix(y_test, pred))


              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2

[[1 0]
 [1 0]]


## Inspect the decision rules


In [6]:
print(export_text(clf, feature_names=features))


|--- cross_border_activity <= 0.50
|   |--- class: 0
|--- cross_border_activity >  0.50
|   |--- class: 1



## Apply to all customers


In [7]:
df['predicted_edd'] = clf.predict(X)
df['predicted_label'] = df['predicted_edd'].map({0:'Standard',1:'EDD'})
df[['customer_id','customer_name','crr_score_0_100','edd_target','predicted_label']]


,customer_id,customer_name,crr_score_0_100,edd_target,predicted_label
0,H001,Atlas Import Group,1,1,EDD
1,H002,Bright Side Retail,0,0,Standard
2,H003,Cedar Family Office,1,1,EDD
3,H004,Delta Community Health,0,0,Standard
4,H005,Evergreen Payments LLC,1,1,EDD
5,H006,Frontier Logistics,0,1,Standard


## Solution summary
The decision tree learns the same policy used to label the data: higher CRR scores and trigger flags route customers to EDD.
